# 13-тарау. Кішкентай GPT-ті қазақ мәтінінде оқытамыз

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 13.1. Жобаның мақсаты

## 13.2. 1-қадам. Мәтін және сөздік

In [ ]:
import urllib.request
import torch

RAW = ("https://raw.githubusercontent.com/amirtemir1996n3-dotcom/"
       "mashina-qalai-uirenedi/main/")
for path in ["data/kitap.txt", "vol2/gpt.py"]:
    urllib.request.urlretrieve(RAW + path, path.split("/")[-1])
from gpt import GPT, GPTConfig

text = open("kitap.txt", encoding="utf-8").read()
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print("Сөздік:", len(chars), "таңба")
print("".join(chars[13:60]))
print(encode("Сәлем"), decode(encode("Сәлем")))

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(len(train_data), len(val_data))

## 13.3. 2-қадам. Оқу мысалдары

In [ ]:
block_size, batch_size = 64, 32

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x, y

torch.manual_seed(0)
x, y = get_batch("train")
print(x.shape, y.shape)
print(repr(decode(x[0, :24].tolist())))
print(repr(decode(y[0, :24].tolist())))

## 13.4. 3-қадам. Модель және оқыту

In [ ]:
@torch.no_grad()
def estimate_loss(model, iters=20):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = [model(*get_batch(split))[1].item() for _ in range(iters)]
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

torch.manual_seed(0)
cfg = GPTConfig(vocab_size=len(chars), block_size=block_size,
                n_layer=4, n_head=4, n_embd=128, dropout=0.1)
model = GPT(cfg)
print(f"Параметрлер: {sum(p.numel() for p in model.parameters()):,}"
      .replace(",", " "))

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
history = []
for step in range(2501):
    if step % 500 == 0:
        losses = estimate_loss(model)
        history.append((step, losses["train"], losses["val"]))
        print(f"{step:4d}: оқу={losses['train']:.3f}, "
              f"валидация={losses['val']:.3f}")
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

## 13.5. 4-қадам. Модель не үйренді?

In [ ]:
model.eval()
ctx = torch.tensor([encode("Жасанды интеллек")])
with torch.no_grad():
    probs = torch.softmax(model(ctx)[0][0, -1], dim=-1)
top = torch.topk(probs, 3)
print([(itos[i.item()], round(p.item(), 3))
       for p, i in zip(top.values, top.indices)])

In [ ]:
torch.manual_seed(1)
ctx = torch.tensor([encode("Нейрондық желі")])
out = model.generate(ctx, max_new_tokens=300, temperature=0.8)
print(decode(out[0].tolist()))

## 13.6. 5-қадам. Модельді сақтау

In [ ]:
torch.save(model.state_dict(), "gpt_kitap.pt")
loaded = GPT(cfg)
loaded.load_state_dict(torch.load("gpt_kitap.pt"))
loaded.eval()
same = torch.equal(loaded(ctx)[0], model(ctx)[0])
print("Модель сақталды:", same)

## 13.7. Модельді үлкейту не береді?